In [1]:
import pandas as pd
import numpy as np
import argparse
import time
from copy import deepcopy
import os
import warnings

from sklearn.metrics import classification_report, f1_score, accuracy_score,precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import WeightedRandomSampler
import math
from tqdm import tqdm
warnings.filterwarnings('ignore')


# 1 feature map
def makemap(df0, r, e=1e-6): 
    df = np.array(deepcopy(df0))    
    n, m = df.shape
    f = int(m / r) 
    
    df = np.hstack([df[:, j * f:(j + 1) * f] /
                    df[:, j * f:(j + 1) * f].sum(1).reshape(-1, 1) for j in range(r)])
    
    data = np.zeros((n, 1, m, m), dtype=np.float)

    for i in range(n):
        for x in range(m):
            for y in range(m):
                if x < y:
                    data[i, 0, x, y] = df[i, x] - df[i, y]
                if x > y:
                    data[i, 0, x, y] = np.tanh(np.log2(((df[i, x] + e) / (df[i, y] + e))))
                if x == y:
                    data[i, 0, x, y] = df[i, x] - 1 / f
    return data.round(3)




class channel_attention(nn.Module):
    def __init__(self, channel, ratio=2):
        super(channel_attention, self).__init__()
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // ratio, False),
            nn.ReLU(),
            nn.Linear(channel // ratio, channel, False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        max_pool_out = self.max_pool(x).view([b, c])
        avg_pool_out = self.avg_pool(x).view([b, c])

        max_fc_out = self.fc(max_pool_out)
        avg_fc_out = self.fc(avg_pool_out)

        out = max_fc_out + avg_fc_out
        out = self.sigmoid(out).view([b, c, 1, 1])
        # print(out)

        return out * x


class spacial_attention(nn.Module):
    def __init__(self, kernel_size=3):
        super(spacial_attention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, 1, padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_pool_out, _ = torch.max(x, dim=1, keepdim=True)
        avg_pool_out = torch.mean(x, dim=1, keepdim=True)
        pool_out = torch.cat([max_pool_out, avg_pool_out], dim=1)

        out = self.conv(pool_out)
        out = self.sigmoid(out)
        # print(out)

        return out * x


class DeepSP(nn.Module):
    def __init__(self, fraction, out_channels, channel=32, ratio=2, kernel_size=3):
        super(DeepSP, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(nn.Conv2d(16, 32, 3, 1, 1),
                                   nn.BatchNorm2d(32), nn.ReLU())

        self.channel_attention = channel_attention(channel, ratio)
        self.spacial_attention = spacial_attention(kernel_size)

        self.fc1 = nn.Sequential(
            nn.Linear(32 * (fraction // 2) ** 2 , 512),
            nn.ReLU(),
            nn.Dropout(p=0.3))

        self.fc2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3))

        self.fc3 = nn.Linear(256, out_channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.channel_attention(x)
        x = self.spacial_attention(x)
        x = self.fc1(x.view(x.size(0), -1))
        x = self.fc2(x)
        x = self.fc3(x)
        return x

class DeepSP_noatt(nn.Module):
    def __init__(self, fraction, out_channels, channel=32, ratio=2, kernel_size=3):
        super(DeepSP_noatt, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=1,
            out_channels=16,
            kernel_size=3,
            stride=1,
            padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(nn.Conv2d(16, 32, 3, 1, 1),
                                   nn.BatchNorm2d(32), nn.ReLU())

#         self.channel_attention = channel_attention(channel, ratio)
#         self.spacial_attention = spacial_attention(kernel_size)

        self.fc1 = nn.Sequential(
            nn.Linear(32 * (fraction // 2) ** 2 , 512),
            nn.ReLU(),
            nn.Dropout(p=0.3))

        self.fc2 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3))

        self.fc3 = nn.Linear(256, out_channels)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
#         x = self.channel_attention(x)
#         x = self.spacial_attention(x)
        x = self.fc1(x.view(x.size(0), -1))
        x = self.fc2(x)
        x = self.fc3(x)
        return x
    
    
    
    
class MultiFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=0):
        super(MultiFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, outputs, targets):
        if self.alpha is None and self.gamma == 0:
            focal_loss = F.cross_entropy(outputs, targets)

        elif self.alpha is not None and self.gamma == 0:
            self.alpha = torch.tensor(self.alpha) / torch.tensor(self.alpha).sum()
            self.alpha = self.alpha.to(outputs)
            focal_loss = F.cross_entropy(outputs, targets, weight=self.alpha)

        elif self.alpha is None and self.gamma != 0:
            ce_loss = F.cross_entropy(outputs, targets, reduction='none')
            p_t = torch.exp(-ce_loss)
            focal_loss = ((1 - p_t) ** self.gamma * ce_loss).mean()

        else:
            self.alpha = torch.tensor(self.alpha) / torch.tensor(self.alpha).sum()
            self.alpha = self.alpha.to(outputs)
            ce_loss = F.cross_entropy(outputs, targets, reduction='none')
            p_t = torch.exp(-ce_loss)
            ce_loss = F.cross_entropy(outputs, targets, weight=self.alpha, reduction='none')
            focal_loss = ((1 - p_t) ** self.gamma * ce_loss).mean()

        return focal_loss

def train(model, train_loader, val_db, index, weight):
    x_val, y_val = val_db
    sm = nn.Softmax(dim=1)
    loss_func = MultiFocalLoss(alpha=weight, gamma=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    val_f1score_best = 0.0
    bestepoch = 0
    for epoch in range(1, epochs + 1):
        model.train()
        for i, (x_batch, y_batch) in enumerate(train_loader):
            out = model(x_batch)
            loss = loss_func(out, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(x_val)
            loss = loss_func(val_out, y_val)
            val_loss = loss.item()
            y_valprob = sm(val_out)
            y_valscore, y_valpred = torch.max(y_valprob, 1)
            val_f1score = f1_score(y_val, y_valpred, average='macro')
            val_acc = accuracy_score(y_val, y_valpred)

            if val_f1score_best < val_f1score:
                bestepoch = epoch
                val_f1score_best = val_f1score
                val_acc_best = val_acc
                val_loss_best = val_loss
                y_valprob_best = y_valprob
                torch.save(model, 'model{}.pkl'.format(index + 1))
    return y_valprob_best.numpy()


def cross_validation(x, y):
    y_trainprob = np.zeros((y.shape[0], len(set(target))))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cv_index = np.zeros(y.shape[0])
    for index, (train_index, val_index) in enumerate(skf.split(x, y)):
        x_train, x_val = x[train_index], x[val_index]
        y_train, y_val = y[train_index], y[val_index]
        train_db = TensorDataset(x_train, y_train)
        weight = 1 / np.bincount(y_train)
        train_loader = DataLoader(train_db,
                                  batch_size=batch_size,
                                  num_workers=0,
                                  shuffle=True)
        val_db = (x_val, y_val)
        torch.manual_seed(seed)
        model = DeepSP(fraction, len(set(target)))
        
        y_trainprob[val_index] = train(model, train_loader, val_db, index, weight)
        cv_index[val_index] = index + 1
    return y_trainprob, cv_index


def cross_validation_noatt(x, y):
    y_trainprob = np.zeros((y.shape[0], len(set(target))))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cv_index = np.zeros(y.shape[0])
    for index, (train_index, val_index) in enumerate(skf.split(x, y)):
        x_train, x_val = x[train_index], x[val_index]
        y_train, y_val = y[train_index], y[val_index]
        train_db = TensorDataset(x_train, y_train)
        weight = 1 / np.bincount(y_train)
        train_loader = DataLoader(train_db,
                                  batch_size=batch_size,
                                  num_workers=0,
                                  shuffle=True)
        val_db = (x_val, y_val)
        torch.manual_seed(seed)
        model = DeepSP_noatt(fraction, len(set(target)))
        y_trainprob[val_index] = train(model, train_loader, val_db, index, weight)
        cv_index[val_index] = index + 1
    return y_trainprob, cv_index



def predict(x_test):
    sm = nn.Softmax(dim=1)
    y_testprob_result = np.zeros((x_test.shape[0], len(set(target))))
    for index in range(n_splits):
        model = torch.load('model{}.pkl'.format(index + 1))
        model.eval()
        with torch.no_grad():
            y_testprob_result += sm(model(x_test)).numpy() / n_splits
        os.remove('model{}.pkl'.format(index + 1))
    return y_testprob_result

# WithoutAttention

In [2]:
datafile = 'nikolovski2014'
rep = 2

n_splits = 5
batch_size = 64
epochs = 100
lr = 1e-3
l2 = 0.0
seed=0

data = pd.read_csv('../{}.csv'.format(datafile), index_col=0)

test_size = 0.2

data = data[data['markers'] != 'unknown']
target = data.markers

class_mapping = {label: idx for idx, label in enumerate(sorted(set(target)))}
inv_class_mapping = {v: k for k, v in class_mapping.items()}


test_set = pd.concat([data[data.markers == i].sample(math.ceil(data[data.markers == i].shape[0] * test_size))
        for i in pd.unique(data.markers)],axis=0)
train_set = data[~data.index.isin(test_set.index)]

train_data = train_set.drop('markers', axis=1)
train_y = train_set.markers

test_data = test_set.drop('markers', axis=1)
test_y = test_set.markers

x_train = torch.tensor(makemap(train_data.values, rep), dtype=torch.float)
y_train = torch.tensor(train_y.map(class_mapping).values)
fraction = x_train.shape[2]


y_trainprob, cv_index = cross_validation_noatt(x_train, y_train)
y_trainpred = y_trainprob.argmax(1)
cv_result_noatt = pd.DataFrame({'cv_index': cv_index, 
                          'true_label': train_y,
                         'pred_label': pd.Series(y_trainpred).map(inv_class_mapping).tolist(),
                         'pred_prob': y_trainprob.max(1)}, index=train_data.index)


x_test = torch.tensor(makemap(test_data.values, rep), dtype=torch.float)
y_testprob = predict(x_test)
y_testpred = y_testprob.argmax(1)
test_result_noatt = pd.DataFrame({'true_label': test_y,
              'pred_label': pd.Series(y_testpred).map(inv_class_mapping).tolist(),
              'pred_prob': y_testprob.max(1)}, index=test_data.index)


In [3]:
def calcmetrics(true_label, pred_label):
    metric = []
    metric.append(precision_score(true_label,pred_label,average='macro'))
    metric.append(recall_score(true_label,pred_label,average='macro'))
    metric.append(f1_score(true_label,pred_label,average='macro'))
    metric.append(accuracy_score(true_label,pred_label))
    return metric

In [4]:
metricresult_noatt = []
for i in range(1, 6):
    metricresult_noatt.append(calcmetrics(cv_result_noatt[cv_result_noatt['cv_index'] == i].true_label,
                                        cv_result_noatt[cv_result_noatt['cv_index'] == i].pred_label))
metricresult_noatt.append(np.array(metricresult_noatt).mean(0).tolist())
metricresult_noatt.append(calcmetrics(test_result_noatt.true_label,test_result_noatt.pred_label))

metricresult_noatt = pd.DataFrame(np.array(metricresult_noatt),
                            columns=['Precision', 'Recall', 'F1-score', 'Accuracy'])
metricresult_noatt.insert(0, 'data', datafile)
metricresult_noatt.insert(1, 'method', 'WithoutAttention')
metricresult_noatt.insert(2, 'result', ['Cross_validation_{}'.format(i) for i in range(1, 6)] + 
                                 ['Cross_validation'] + ['Test'])
metricresult_noatt

,data,method,result,Precision,Recall,F1-score,Accuracy
0,nikolovski2014,WithoutAttention,Cross_validation_1,0.795833,0.800000,0.795169,0.775000
1,nikolovski2014,WithoutAttention,Cross_validation_2,0.763826,0.727083,0.731376,0.725000
2,nikolovski2014,WithoutAttention,Cross_validation_3,0.844097,0.853125,0.845997,0.820513
3,nikolovski2014,WithoutAttention,Cross_validation_4,0.868750,0.881250,0.866830,0.846154
4,nikolovski2014,WithoutAttention,Cross_validation_5,0.708681,0.721875,0.686000,0.717949
5,nikolovski2014,WithoutAttention,Cross_validation,0.796237,0.796667,0.785074,0.776923
6,nikolovski2014,WithoutAttention,Test,0.697642,0.666771,0.670194,0.672727


# WithAttention

In [5]:
y_trainprob, cv_index = cross_validation(x_train, y_train)
y_trainpred = y_trainprob.argmax(1)
cv_result_att = pd.DataFrame({'cv_index': cv_index, 
                          'true_label': train_y,
                         'pred_label': pd.Series(y_trainpred).map(inv_class_mapping).tolist(),
                         'pred_prob': y_trainprob.max(1)}, index=train_data.index)

x_test = torch.tensor(makemap(test_data.values, rep), dtype=torch.float)
y_testprob = predict(x_test)
y_testpred = y_testprob.argmax(1)
test_result_att = pd.DataFrame({'true_label': test_y,
              'pred_label': pd.Series(y_testpred).map(inv_class_mapping).tolist(),
              'pred_prob': y_testprob.max(1)}, index=test_data.index)

In [6]:
metricresult_att = []
for i in range(1, 6):
    metricresult_att.append(calcmetrics(cv_result_att[cv_result_att['cv_index'] == i].true_label,
                                        cv_result_att[cv_result_att['cv_index'] == i].pred_label))
metricresult_att.append(np.array(metricresult_att).mean(0).tolist())
metricresult_att.append(calcmetrics(test_result_att.true_label,test_result_att.pred_label))

metricresult_att = pd.DataFrame(np.array(metricresult_att),
                            columns=['Precision', 'Recall', 'F1-score', 'Accuracy'])
metricresult_att.insert(0, 'data', datafile)
metricresult_att.insert(1, 'method', 'WithAttention')
metricresult_att.insert(2, 'result', ['Cross_validation_{}'.format(i) for i in range(1, 6)] + 
                                 ['Cross_validation'] + ['Test'])
metricresult_att

,data,method,result,Precision,Recall,F1-score,Accuracy
0,nikolovski2014,WithAttention,Cross_validation_1,0.886458,0.862500,0.858555,0.850000
1,nikolovski2014,WithAttention,Cross_validation_2,0.783333,0.758333,0.758460,0.750000
2,nikolovski2014,WithAttention,Cross_validation_3,0.845076,0.803125,0.817460,0.794872
3,nikolovski2014,WithAttention,Cross_validation_4,0.840972,0.818750,0.824248,0.794872
4,nikolovski2014,WithAttention,Cross_validation_5,0.725321,0.734375,0.694311,0.743590
5,nikolovski2014,WithAttention,Cross_validation,0.816232,0.795417,0.790607,0.786667
6,nikolovski2014,WithAttention,Test,0.758681,0.705461,0.716510,0.709091


In [7]:
metric_result = pd.concat([metricresult_noatt, metricresult_att], axis=0)
metric_result

,data,method,result,Precision,Recall,F1-score,Accuracy
0,nikolovski2014,WithoutAttention,Cross_validation_1,0.795833,0.800000,0.795169,0.775000
1,nikolovski2014,WithoutAttention,Cross_validation_2,0.763826,0.727083,0.731376,0.725000
2,nikolovski2014,WithoutAttention,Cross_validation_3,0.844097,0.853125,0.845997,0.820513
3,nikolovski2014,WithoutAttention,Cross_validation_4,0.868750,0.881250,0.866830,0.846154
4,nikolovski2014,WithoutAttention,Cross_validation_5,0.708681,0.721875,0.686000,0.717949
5,nikolovski2014,WithoutAttention,Cross_validation,0.796237,0.796667,0.785074,0.776923
6,nikolovski2014,WithoutAttention,Test,0.697642,0.666771,0.670194,0.672727
0,nikolovski2014,WithAttention,Cross_validation_1,0.886458,0.862500,0.858555,0.850000
1,nikolovski2014,WithAttention,Cross_validation_2,0.783333,0.758333,0.758460,0.750000
2,nikolovski2014,WithAttention,Cross_validation_3,0.845076,0.803125,0.817460,0.794872


In [9]:
cv_result_noatt.to_csv('{}_CVresult_noatt.csv'.format(datafile), index=None)
test_result_noatt.to_csv('{}_Testresult_noatt.csv'.format(datafile), index=None)
cv_result_att.to_csv('{}_CVresult_att.csv'.format(datafile), index=None)
test_result_att.to_csv('{}_Testresult_att.csv'.format(datafile), index=None)
metric_result.to_csv('{}_AttentionModule.csv'.format(datafile), index=None)